# Meta-Learning

## Learning Objectives
1. Understand the MAML inner/outer-loop optimisation framework.
2. Implement a MAML-style gradient-update loop from scratch with NumPy.
3. Build prototypical networks in PyTorch for few-shot classification.
4. Compare zero-shot vs few-shot generalisation strategies in production.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from collections import defaultdict

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: MAML Inner / Outer Loop (NumPy)


In [ ]:
def task_loss(params, x, y):
    """Linear regression loss for a single task (used as inner-loop loss)."""
    w, b = params
    pred = x @ w + b
    return np.mean((pred - y) ** 2)


def task_gradient(params, x, y):
    """Analytic gradient of MSE w.r.t. [w, b]."""
    w, b = params
    pred = x @ w + b
    err = pred - y          # (n, 1)
    dw = (2 / len(x)) * x.T @ err
    db = (2 / len(x)) * err.mean()
    return [dw, db]


def maml_step(meta_params, tasks, inner_lr=0.01, outer_lr=0.001, n_inner=1):
    """Single MAML outer-step over a batch of tasks."""
    meta_w, meta_b = meta_params
    outer_grads_w = np.zeros_like(meta_w)
    outer_grads_b = 0.0

    for (x_s, y_s, x_q, y_q) in tasks:
        # --- inner loop: adapt on support set ---
        w, b = meta_w.copy(), meta_b
        for _ in range(n_inner):
            gw, gb = task_gradient([w, b], x_s, y_s)
            w = w - inner_lr * gw
            b = b - inner_lr * gb

        # --- outer loop: compute query-set gradient w.r.t adapted params ---
        gw_q, gb_q = task_gradient([w, b], x_q, y_q)
        outer_grads_w += gw_q
        outer_grads_b += gb_q

    outer_grads_w /= len(tasks)
    outer_grads_b /= len(tasks)

    # --- meta-update ---
    new_w = meta_w - outer_lr * outer_grads_w
    new_b = meta_b - outer_lr * outer_grads_b
    return [new_w, new_b]


def make_sinusoidal_tasks(n_tasks=4, n_support=5, n_query=10, d=1):
    tasks = []
    for _ in range(n_tasks):
        amplitude = np.random.uniform(0.5, 2.0)
        phase = np.random.uniform(0, np.pi)
        x = np.random.uniform(-3, 3, (n_support + n_query, d))
        y = amplitude * np.sin(x + phase)
        tasks.append((x[:n_support], y[:n_support], x[n_support:], y[n_support:]))
    return tasks


d = 1
meta_params = [np.random.randn(d, 1) * 0.1, 0.0]
meta_losses = []
for step in range(200):
    tasks = make_sinusoidal_tasks(n_tasks=4, d=d)
    meta_params = maml_step(meta_params, tasks, inner_lr=0.05, outer_lr=0.002)
    if step % 50 == 0:
        test_tasks = make_sinusoidal_tasks(n_tasks=10, d=d)
        avg_loss = np.mean([
            task_loss(meta_params, x_q, y_q) for _, _, x_q, y_q in test_tasks
        ])
        meta_losses.append(avg_loss)
        print(f"Step {step:3d} | meta query-loss: {avg_loss:.4f}")

print("Final meta-params (w, b):", meta_params[0].ravel(), meta_params[1])


## Level 2: Prototypical Networks (PyTorch + OOM Handling)


In [ ]:
class ProtoNet(nn.Module):
    """Simple embedding network for prototypical few-shot learning."""

    def __init__(self, in_dim: int = 64, hidden: int = 128, out_dim: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)


def prototypical_loss(support_embeddings, support_labels,
                      query_embeddings, query_labels, n_way):
    """Compute prototype loss: euclidean distance to class centroids."""
    prototypes = torch.stack([
        support_embeddings[support_labels == c].mean(0)
        for c in range(n_way)
    ])
    dists = torch.cdist(query_embeddings, prototypes)
    log_p = F.log_softmax(-dists, dim=1)
    return F.nll_loss(log_p, query_labels)


def make_episode(n_way=5, n_support=5, n_query=15, in_dim=64):
    """Synthetic episode for prototypical training."""
    support_x, support_y, query_x, query_y = [], [], [], []
    for c in range(n_way):
        mu = torch.randn(in_dim) * 2
        support_x.append(mu + torch.randn(n_support, in_dim) * 0.5)
        support_y.extend([c] * n_support)
        query_x.append(mu + torch.randn(n_query, in_dim) * 0.5)
        query_y.extend([c] * n_query)
    sx = torch.cat(support_x).to(device)
    sy = torch.tensor(support_y, device=device)
    qx = torch.cat(query_x).to(device)
    qy = torch.tensor(query_y, device=device)
    return sx, sy, qx, qy


model_proto = ProtoNet(in_dim=64).to(device)
opt_proto = torch.optim.Adam(model_proto.parameters(), lr=1e-3)
N_WAY = 5

losses = []
for episode in range(300):
    sx, sy, qx, qy = make_episode(n_way=N_WAY)
    try:
        opt_proto.zero_grad()
        se = model_proto(sx)
        qe = model_proto(qx)
        loss = prototypical_loss(se, sy, qe, qy, N_WAY)
        loss.backward()
        opt_proto.step()
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            print("OOM: reduce n_way or n_query")
            torch.cuda.empty_cache()
            continue
        raise
    if episode % 100 == 0:
        losses.append(loss.item())
        print(f"Episode {episode:3d} | loss: {loss.item():.4f}")

print(f"Final episode loss: {losses[-1]:.4f}")


## Real-World Example 1: MAML 5-Shot Sine Regression


In [ ]:
class SineNet(nn.Module):
    """Small MLP for MAML sine regression."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 40), nn.ReLU(),
            nn.Linear(40, 40), nn.ReLU(),
            nn.Linear(40, 1),
        )
    def forward(self, x):
        return self.net(x)


def clone_params(model):
    """Return detached clone of each parameter tensor."""
    return [p.clone() for p in model.parameters()]


def functional_forward(model, x, params):
    """Manual forward pass with given parameter list (for MAML in PyTorch)."""
    idx = 0
    out = x
    for layer in model.net:
        if isinstance(layer, nn.Linear):
            w, b = params[idx], params[idx + 1]
            out = F.linear(out, w, b)
            idx += 2
        elif isinstance(layer, nn.ReLU):
            out = F.relu(out)
    return out


def sine_task(n_support=5, n_query=10):
    amp = np.random.uniform(0.5, 2.0)
    phase = np.random.uniform(0, np.pi)
    x = torch.FloatTensor(n_support + n_query, 1).uniform_(-5, 5).to(device)
    y = (amp * torch.sin(x + phase)).to(device)
    return x[:n_support], y[:n_support], x[n_support:], y[n_support:]


maml_model = SineNet().to(device)
meta_opt = torch.optim.Adam(maml_model.parameters(), lr=1e-3)
INNER_LR = 0.01

for meta_step in range(500):
    tasks = [sine_task() for _ in range(4)]
    meta_loss = torch.tensor(0.0, device=device)
    for xs, ys, xq, yq in tasks:
        fast_params = clone_params(maml_model)
        for _ in range(5):
            pred = functional_forward(maml_model, xs, fast_params)
            inner_loss = F.mse_loss(pred, ys)
            grads = torch.autograd.grad(inner_loss, fast_params, create_graph=True)
            fast_params = [p - INNER_LR * g for p, g in zip(fast_params, grads)]
        query_pred = functional_forward(maml_model, xq, fast_params)
        meta_loss = meta_loss + F.mse_loss(query_pred, yq)

    meta_opt.zero_grad()
    (meta_loss / len(tasks)).backward()
    meta_opt.step()
    if meta_step % 100 == 0:
        print(f"Meta-step {meta_step:4d} | query MSE: {(meta_loss/len(tasks)).item():.4f}")

print("MAML 5-shot training complete.")


## Real-World Example 2: Meta Domain Adaptation


In [ ]:
class DomainNet(nn.Module):
    """Shared encoder + linear head for domain adaptation."""
    def __init__(self, in_dim=20, hidden=64, n_classes=3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.head = nn.Linear(hidden, n_classes)

    def forward(self, x):
        return self.head(self.encoder(x))


def make_domain_data(n_domains=6, n_per_domain=200, in_dim=20, n_classes=3):
    """Each domain has a different linear shift in feature space."""
    domains = []
    for d_idx in range(n_domains):
        shift = torch.randn(in_dim) * (d_idx * 0.5)
        x = torch.randn(n_per_domain, in_dim) + shift
        y = torch.randint(0, n_classes, (n_per_domain,))
        domains.append(TensorDataset(x, y))
    return domains


domains = make_domain_data()
target_domain = domains[-1]
source_domains = domains[:-1]

meta_da_model = DomainNet().to(device)
meta_da_opt = torch.optim.Adam(meta_da_model.parameters(), lr=1e-3)
crit_da = nn.CrossEntropyLoss()

for episode in range(400):
    d_idx = np.random.randint(len(source_domains))
    loader = DataLoader(source_domains[d_idx], batch_size=32, shuffle=True)
    episode_loss = 0.0
    meta_da_model.train()
    for xb, yb in loader:
        meta_da_opt.zero_grad()
        try:
            loss = crit_da(meta_da_model(xb.to(device)), yb.to(device))
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                print("OOM: reduce batch size"); torch.cuda.empty_cache(); continue
            raise
        loss.backward(); meta_da_opt.step()
        episode_loss += loss.item()
    if episode % 100 == 0:
        print(f"Episode {episode:3d} | source loss: {episode_loss/len(loader):.4f}")

# Fine-tune on 10 labelled examples from target domain
fine_tune_data = TensorDataset(
    target_domain.tensors[0][:10], target_domain.tensors[1][:10]
)
fine_tune_loader = DataLoader(fine_tune_data, batch_size=10)
fine_tune_opt = torch.optim.Adam(meta_da_model.parameters(), lr=5e-4)
for _ in range(50):
    for xb, yb in fine_tune_loader:
        fine_tune_opt.zero_grad()
        crit_da(meta_da_model(xb.to(device)), yb.to(device)).backward()
        fine_tune_opt.step()

meta_da_model.eval()
with torch.no_grad():
    xt, yt = target_domain.tensors
    preds = meta_da_model(xt.to(device)).argmax(1).cpu()
    acc = (preds == yt).float().mean().item()
print(f"Target domain accuracy after meta-DA + 10-shot fine-tune: {acc:.3f}")


## Real-World Example 3: Zero-Shot vs Few-Shot Generalisation Comparison


In [ ]:
import copy

def evaluate_k_shot(base_model, target_ds, k, n_classes=3, n_finetune_steps=50):
    """Fine-tune on k examples per class, evaluate on rest."""
    model_k = copy.deepcopy(base_model).to(device)
    opt_k = torch.optim.Adam(model_k.parameters(), lr=5e-4)
    crit_k = nn.CrossEntropyLoss()

    xs, ys = target_ds.tensors
    support_x, support_y = [], []
    for c in range(n_classes):
        idx = (ys == c).nonzero(as_tuple=True)[0][:k]
        support_x.append(xs[idx]); support_y.append(ys[idx])
    sup_x = torch.cat(support_x).to(device)
    sup_y = torch.cat(support_y).to(device)

    for _ in range(n_finetune_steps):
        opt_k.zero_grad()
        crit_k(model_k(sup_x), sup_y).backward()
        opt_k.step()

    model_k.eval()
    with torch.no_grad():
        preds = model_k(xs.to(device)).argmax(1).cpu()
    return (preds == ys).float().mean().item()


k_shots = [0, 1, 5, 10, 20]
accuracies = []
for k in k_shots:
    if k == 0:
        meta_da_model.eval()
        with torch.no_grad():
            xt, yt = target_domain.tensors
            preds = meta_da_model(xt.to(device)).argmax(1).cpu()
        acc = (preds == yt).float().mean().item()
    else:
        acc = evaluate_k_shot(meta_da_model, target_domain, k)
    accuracies.append(acc)
    print(f"k={k:2d}-shot accuracy: {acc:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_shots, accuracies, marker='o', linewidth=2, color='steelblue')
ax.set_xlabel("Number of Fine-Tune Examples (k-shot)")
ax.set_ylabel("Target Domain Accuracy")
ax.set_title("Zero-Shot vs Few-Shot Generalisation")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/meta_learning_kshot.png", dpi=80)
plt.close()
print("Saved comparison plot to /tmp/meta_learning_kshot.png")
